[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/upm-classes/ai_medical_imaging_2026/blob/main/notebooks/student/NB04_baseline_cnn.ipynb)

# NB04 · Baseline CNN, honest metrics and the tuning-race kit
**Artificial Intelligence in Medical Imaging** · Day 2 · Lab 5 (30 min) and tuning race (25 min)

| | |
|---|---|
| Runtime | CPU is enough (a few seconds per epoch) |
| Data | PneumoniaMNIST 28 px (4,708 / 524 / 624), CC BY 4.0 |
| Checkpoints | output shape (batch, 2) · 2,050 parameters · accuracy above the majority-class rate · metrics equal scikit-learn's |
| Tiers | **Core** = TODO cells · **Plus** = error hunt, number of filters · **Challenge** = third block + BatchNorm |

**Learning goals.** (1) Build the CNN of the Day-2 slides and verify its shapes and parameter count. (2) Train it with a train/validation loop and read the learning curves. (3) Compute the confusion matrix, precision, recall and F1 by hand. (4) Compare against a simple baseline *and against the noise*.

**Convention in this notebook.** The network outputs **two scores (logits)**, one per class, and we use `CrossEntropyLoss` with integer labels. (NB03b used one score and `BCEWithLogitsLoss`. Both are standard for two classes; the slide's CNN ends in 400 → 2.) The test set stays locked.

In [ ]:
%pip install -q "medmnist==3.0.2"
import os, random, time, math
from pathlib import Path
import numpy as np, matplotlib.pyplot as plt
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

def set_seed(seed=42):
    random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed); torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

SEED = 42
set_seed(SEED)
for d in ("data", "outputs"): Path(d).mkdir(exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device)

In [ ]:
def load_pneumonia():
    """PneumoniaMNIST (28 px, CC BY 4.0). Falls back to SYNTHETIC data if the download fails."""
    try:
        from medmnist import PneumoniaMNIST
        out = {}
        for s in ("train", "val", "test"):
            d = PneumoniaMNIST(split=s, download=True, root="data")
            out[s] = (d.imgs.reshape(len(d.imgs), 28, 28).astype(np.uint8), d.labels.reshape(-1).astype(np.int64))
        return out, True
    except Exception as e:
        print("!! Could not download PneumoniaMNIST:", repr(e))
        print("!! Using SYNTHETIC data so the code still runs. Numbers below are NOT about pneumonia.")
        rng = np.random.default_rng(0); out = {}
        for s, n in (("train", 2000), ("val", 300), ("test", 300)):
            y = (rng.random(n) < 0.7).astype(np.int64)
            x = rng.normal(100, 25, (n, 28, 28)); x[y == 1, 8:20, 8:20] += 40
            out[s] = (x.clip(0, 255).astype(np.uint8), y)
        return out, False

data, REAL = load_pneumonia()
to_tensor = lambda x: ((torch.from_numpy(x).float() / 255.0 - 0.5) / 0.5).unsqueeze(1)    # (N, 1, 28, 28), range -1..1
X = {s: to_tensor(v[0]) for s, v in data.items()}
Y = {s: torch.from_numpy(v[1]) for s, v in data.items()}
print({s: tuple(X[s].shape) for s in X}, "| real data:", REAL)
maj_val = max(Y["val"].float().mean().item(), 1 - Y["val"].float().mean().item())
print(f"majority-class rate on validation: {maj_val:.3f}  (an 'always the same answer' model scores this)")

In [ ]:
def make_loader(split, bs=128, shuffle=False, seed=SEED):
    g = torch.Generator().manual_seed(seed)
    return DataLoader(TensorDataset(X[split], Y[split]), batch_size=bs, shuffle=shuffle, generator=g if shuffle else None)

import torch.nn.functional as F
def random_affine(xb, max_deg=8, max_shift=0.08):
    """Light random rotation (+/- max_deg degrees) and shift (fraction of the image), per image. Used by the race kit."""
    n = xb.size(0)
    ang = (torch.rand(n, device=xb.device) * 2 - 1) * max_deg * math.pi / 180
    tx = (torch.rand(n, device=xb.device) * 2 - 1) * max_shift * 2        # affine_grid coordinates run from -1 to 1
    ty = (torch.rand(n, device=xb.device) * 2 - 1) * max_shift * 2
    c, s = torch.cos(ang), torch.sin(ang)
    theta = torch.stack([torch.stack([c, -s, tx], 1), torch.stack([s, c, ty], 1)], 1)      # (n, 2, 3)
    grid = F.affine_grid(theta, xb.size(), align_corners=False)
    return F.grid_sample(xb, grid, padding_mode="border", align_corners=False)

@torch.no_grad()
def predict(model, split):
    """Return predicted classes, probabilities of class 1 and true labels for a whole split (eval mode)."""
    model.eval(); preds, probs = [], []
    for xb, _ in make_loader(split, bs=512):
        p = torch.softmax(model(xb.to(device)), dim=1)[:, 1].cpu()
        probs.append(p); preds.append((p > 0.5).long())
    return torch.cat(preds).numpy(), torch.cat(probs).numpy(), Y[split].numpy()

def fit(model, epochs=8, lr=1e-3, bs=128, augment=False, seed=SEED, verbose=True, class_weights=None):
    """Train on the training set, evaluate on the validation set after each epoch. The test set is never used here."""
    set_seed(seed)
    model.to(device)
    train_loader = make_loader("train", bs=bs, shuffle=True, seed=seed)
    val_loader = make_loader("val", bs=512)
    w = None if class_weights is None else torch.tensor(class_weights, dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=w)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    h = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    for ep in range(epochs):
        model.train(); tl, tc, n = 0.0, 0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            if augment: xb = random_affine(xb)              # training images only
            opt.zero_grad(); out = model(xb); loss = criterion(out, yb); loss.backward(); opt.step()
            tl += loss.item() * len(xb); tc += (out.argmax(1) == yb).sum().item(); n += len(xb)
        model.eval(); vl, vc, vn = 0.0, 0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device); out = model(xb)
                vl += criterion(out, yb).item() * len(xb); vc += (out.argmax(1) == yb).sum().item(); vn += len(xb)
        h["train_loss"].append(tl / n); h["train_acc"].append(tc / n); h["val_loss"].append(vl / vn); h["val_acc"].append(vc / vn)
        if verbose: print(f"epoch {ep+1:2d} | train loss {tl/n:.3f} acc {tc/n:.3f} | val loss {vl/vn:.3f} acc {vc/vn:.3f}")
    return h

def plot_curves(h, title=""):
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
    ax[0].plot(h["train_loss"], label="train"); ax[0].plot(h["val_loss"], label="validation"); ax[0].set_title("loss " + title); ax[0].set_xlabel("epoch"); ax[0].legend()
    ax[1].plot(h["train_acc"], label="train"); ax[1].plot(h["val_acc"], label="validation"); ax[1].axhline(maj_val, color="grey", ls="--"); ax[1].set_title("accuracy " + title); ax[1].set_xlabel("epoch"); ax[1].legend()
    plt.tight_layout(); plt.show()
print("helpers ready: make_loader, predict, fit, plot_curves, random_affine")

## 1. Build the CNN of the slides (8 min)
Input 1 × 28 × 28 → `Conv(1→8, 3×3)` → ReLU → `MaxPool(2)` → `Conv(8→16, 3×3)` → ReLU → `MaxPool(2)` → Flatten → `Linear(?, 2)`.

**Predict before you code:** the shape after each layer, and the number that goes into `Linear`. (Slide: 8 × 26 × 26, 8 × 13 × 13, 16 × 11 × 11, 16 × 5 × 5, then 400.) What changes with `padding=1`?

**TODO 1.** Write the model.

In [ ]:
cnn = nn.Sequential(
    # TODO 1a: Conv2d(1, 8, kernel_size=3), ReLU, MaxPool2d(2)

    # TODO 1b: Conv2d(8, 16, kernel_size=3), ReLU, MaxPool2d(2)

    nn.Flatten(),
    # TODO 1c: Linear(?, 2)  -- how many features come out of Flatten? (hint: 16 x 5 x 5)
)

In [ ]:
x = torch.zeros(4, 1, 28, 28)
for layer in cnn:
    x = layer(x); print(f"{layer.__class__.__name__:10s} -> {tuple(x.shape)}")
for name, p in cnn.named_parameters(): print(f"{name:8s} {tuple(p.shape)}  {p.numel()}")
n_params = sum(p.numel() for p in cnn.parameters())
print("total parameters:", n_params)

### Checkpoint 1 · shapes and parameter count

In [ ]:
assert x.shape == (4, 2), "the output must be (batch, 2): one score per class"
assert n_params == 2050, f"expected 2,050 = 80 + 1,168 + 802 as on the slide, got {n_params}"
print("Checkpoint 1 passed: 80 + 1,168 + 802 = 2,050")

## 2. Train and read the curves (12 min)

In [ ]:
cnn_core = cnn
h_cnn = fit(cnn_core, epochs=8)
plot_curves(h_cnn, "(CNN)")

In [ ]:
assert h_cnn["train_loss"][-1] < h_cnn["train_loss"][0], "training loss should fall"
if REAL: assert h_cnn["val_acc"][-1] > maj_val, f"validation accuracy {h_cnn['val_acc'][-1]:.3f} is not above the majority-class rate {maj_val:.3f}: check the learning rate and the loop"
print(f"Checkpoint 2 passed: val acc {h_cnn['val_acc'][-1]:.3f} vs majority {maj_val:.3f}")

**Look at your curves** (keep this plot open for the overfitting talk): is the gap between training and validation accuracy growing? Where would you stop?

## 3. Beyond accuracy: the confusion matrix (10 min)
Positive = pneumonia (label 1). For every image, the model's answer falls into one cell:

| | predicted normal | predicted pneumonia |
|---|---|---|
| **truly normal** | TN | FP |
| **truly pneumonia** | FN | TP |

**TODO 2.** Complete `confusion_counts` and `metrics`. Definitions: precision = TP/(TP+FP) · recall (sensitivity) = TP/(TP+FN) · specificity = TN/(TN+FP) · F1 = harmonic mean of precision and recall.

In [ ]:
def confusion_counts(y_true, y_pred):
    # TODO 2a: count the four cells. Example: tn = images with y_true == 0 AND y_pred == 0
    tn = fp = fn = tp = 0
    return tn, fp, fn, tp

def metrics(tn, fp, fn, tp):
    div = lambda a, b: a / b if b else 0.0            # avoids dividing by zero
    # TODO 2b: accuracy, precision, recall, specificity and f1 from the four counts
    acc = prec = rec = spec = f1 = 0.0
    return dict(accuracy=acc, precision=prec, recall=rec, specificity=spec, f1=f1)

In [ ]:
pred_cnn, prob_cnn, y_val = predict(cnn_core, "val")
cm = confusion_counts(y_val, pred_cnn)
m_cnn = metrics(*cm)
print("TN FP FN TP:", cm)
print({k: round(v, 3) for k, v in m_cnn.items()})
plt.imshow([[cm[0], cm[1]], [cm[2], cm[3]]], cmap="Blues")
for i in range(2):
    for j in range(2): plt.text(j, i, [[cm[0], cm[1]], [cm[2], cm[3]]][i][j], ha="center", va="center", fontsize=14)
plt.xticks([0, 1], ["pred normal", "pred pneumonia"]); plt.yticks([0, 1], ["true normal", "true pneumonia"]); plt.title("validation confusion matrix"); plt.show()

### Checkpoint 3 · your numbers equal scikit-learn's

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score
tn, fp, fn, tp = confusion_matrix(y_val, pred_cnn).ravel()
assert (tn, fp, fn, tp) == cm, "confusion counts differ from scikit-learn"
for name, ref in [("accuracy", accuracy_score(y_val, pred_cnn)), ("precision", precision_score(y_val, pred_cnn, zero_division=0)),
                  ("recall", recall_score(y_val, pred_cnn)), ("f1", f1_score(y_val, pred_cnn))]:
    assert abs(m_cnn[name] - ref) < 1e-9, f"{name} differs from scikit-learn"
assert abs(m_cnn["specificity"] - tn / (tn + fp)) < 1e-9
print("Checkpoint 3 passed")

## 4. A baseline to beat
A result means something only against a simple alternative. Logistic regression on the pixels (two scores, same loss and pipeline as the CNN):

In [ ]:
baseline = nn.Sequential(nn.Flatten(), nn.Linear(784, 2))
h_base = fit(baseline, epochs=8, verbose=False)
pred_b, prob_b, _ = predict(baseline, "val")
m_base = metrics(*confusion_counts(y_val, pred_b))
n_val = len(y_val)
print(f"{'':22s} {'accuracy':>9s} {'precision':>10s} {'recall':>7s} {'specificity':>12s} {'F1':>6s}")
for name, m in [("logistic regression", m_base), ("CNN (2,050 params)", m_cnn)]:
    print(f"{name:22s} {m['accuracy']:9.3f} {m['precision']:10.3f} {m['recall']:7.3f} {m['specificity']:12.3f} {m['f1']:6.3f}")
se = (m_cnn["accuracy"] * (1 - m_cnn["accuracy"]) / n_val) ** 0.5
print(f"majority-class rate {maj_val:.3f} | noise of one accuracy on {n_val} images: +/- {100*se:.1f} points")
import json; Path("outputs/nb04_baselines.json").write_text(json.dumps({"logreg_val_acc": m_base["accuracy"], "cnn_val_acc": m_cnn["accuracy"], "majority": maj_val}, indent=1))

**Reflection questions.**
1. Is the CNN clearly better than the baseline? Is the gap larger than the noise? What would you do to find out?
2. Which is worse in a pneumonia *screening* tool, a false negative or a false positive? Which of the numbers above tells you how many of each you have?
3. The model says "pneumonia" for 95% of images and gets 90% accuracy. What do precision, recall and specificity look like? (Try it: `confusion_counts` on an all-ones prediction.)
4. Why can the validation accuracy be higher than the training accuracy in the first epochs?

In [ ]:
# the "always pneumonia" model
tn_, fp_, fn_, tp_ = confusion_counts(y_val, np.ones_like(y_val))
print({k: round(v, 3) for k, v in metrics(tn_, fp_, fn_, tp_).items()})
assert metrics(tn_, fp_, fn_, tp_)["recall"] == 1.0 and metrics(tn_, fp_, fn_, tp_)["specificity"] == 0.0

## 5. Tuning race kit (used at 14:30, extends this lab)
Your team changes **one factor** of the baseline CNN: learning rate, batch size, depth, or augmentation. **Predict first** (↑ better, ↓ worse, = same), then run, then post your validation accuracy on the board.

Rules: change one thing; fix the seed; compare with the baseline number from section 2 (same seed, same epochs). The test set stays locked.

In [ ]:
def build_cnn(filters=(8, 16), batchnorm=False, n_out=2):
    layers, c, size = [], 1, 28
    for f in filters:
        layers += [nn.Conv2d(c, f, 3)] + ([nn.BatchNorm2d(f)] if batchnorm else []) + [nn.ReLU(), nn.MaxPool2d(2)]
        c, size = f, (size - 2) // 2
    layers += [nn.Flatten(), nn.Linear(c * size * size, n_out)]
    return nn.Sequential(*layers)

assert sum(p.numel() for p in build_cnn().parameters()) == 2050, "build_cnn() must reproduce the 2,050-parameter model"

def run_experiment(name, lr=1e-3, bs=128, filters=(8, 16), batchnorm=False, augment=False, epochs=8, seed=SEED):
    m = build_cnn(filters, batchnorm)
    h = fit(m, epochs=epochs, lr=lr, bs=bs, augment=augment, seed=seed, verbose=False)
    print(f"[{name}] params {sum(p.numel() for p in m.parameters()):,} | final val acc {h['val_acc'][-1]:.3f} | best {max(h['val_acc']):.3f} | final train acc {h['train_acc'][-1]:.3f}")
    return h

# YOUR TEAM: edit ONE argument, write your prediction first
prediction = "?"            # "up", "down" or "same"
h_race = run_experiment("my team", lr=1e-3, bs=128, filters=(8, 16), batchnorm=False, augment=False)
print("prediction:", prediction, "| baseline (section 2):", round(h_cnn["val_acc"][-1], 3), "| this run:", round(h_race["val_acc"][-1], 3))

**After the race:** one validation accuracy on 524 images moves by about ±1.6 points from noise alone. Which differences on the board are bigger than that? (Day 3 gives you confidence intervals to say this properly.)

### Plus · error hunt (the bugs from the Day-2 activity)
The function below runs and prints numbers. It is still wrong. **TODO:** list at least three bugs in `bugs`, then write the corrected function.

In [ ]:
# [PLUS]
def buggy_train(model, epochs=3, lr=1e-3):
    train_loader, test_loader = make_loader("train", shuffle=True), make_loader("test")
    criterion = nn.CrossEntropyLoss(); opt = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        for xb, yb in train_loader:
            out = model(xb.to(device)); loss = criterion(out, yb.to(device))
            loss.backward(); opt.step()
        correct = 0
        for xb, yb in test_loader:
            correct += (model(xb.to(device)).argmax(1).cpu() == yb).sum().item()
        print(f"epoch {epoch+1}: accuracy {correct/len(Y['test']):.3f}")

# TODO (Plus): read buggy_train WITHOUT running it. Write at least three bugs, one sentence each.
bugs = []
assert len(bugs) >= 3 and all(len(b) > 15 for b in bugs), "write at least three bugs (a sentence each)"

### Plus · number of filters
Run `build_cnn(filters=(4, 8))`, `(8, 16)` and `(16, 32)`. Report parameters and validation accuracy. Does the biggest model win by more than the noise?

In [ ]:
# [PLUS]
# TODO (Plus): call run_experiment for filters (4, 8), (8, 16), (16, 32); write down parameters and accuracy
...

### Challenge · a third block and BatchNorm
Compare `filters=(8, 16)` with `filters=(8, 16, 32)` and with `batchnorm=True`. Report parameters, validation accuracy and the train/validation gap. Do not trust a single run: repeat with `seed=1` and `seed=2`.

In [ ]:
# [CHALLENGE]
# TODO (Challenge): for each configuration run seeds 1 and 2, then print mean validation accuracy.
...

## Summary
- The CNN of the slides has 2,050 parameters; shapes are the main source of bugs, so print them.
- Accuracy hides the type of error. Precision, recall and specificity say which error you make.
- A model is a result only if it beats a simple baseline by more than the noise.
- Tune on validation, never on test.

*Next:* NB05 (augmentation and the plausibility audit).